# IPS Portfolio — SIEFORE Básica Generacional 75–79 / Afore XXI Banorte

Asignación objetivo:

| Sleeve | Peso |
|---|---:|
| Mexican government fixed income | 45% |
| Domestic corporate and bank fixed income | 12% |
| International fixed income | 3% |
| Mexican equities | 8% |
| International equities | 15% |
| Structured instruments | 10% |
| FIBRAs | 3% |
| Commodities | 2% |
| Cash and liquid assets | 2% |

## 1. Instalación e imports

In [84]:
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("pandas")
ensure_package("numpy")
ensure_package("openpyxl")
ensure_package("yfinance")
ensure_package("plotly")

import re
import unicodedata
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 2. Parámetros del IPS

In [85]:
# ============================================================
# Parámetros generales del IPS
# ============================================================

TOTAL_PORTFOLIO_MXN = 50_000_000_000  # 50 mil millones MXN

IPS_WEIGHTS = {
    "Mexican government fixed income": 0.45,
    "Domestic corporate and bank fixed income": 0.12,
    "International fixed income": 0.03,
    "Mexican equities": 0.08,
    "International equities": 0.15,
    "Structured instruments": 0.10,
    "FIBRAs": 0.03,
    "Commodities": 0.02,
    "Cash and liquid assets": 0.02,
}

weights_check = pd.DataFrame({
    "Sleeve": list(IPS_WEIGHTS.keys()),
    "Target Weight": list(IPS_WEIGHTS.values())
})
weights_check["Target Weight %"] = weights_check["Target Weight"] * 100
weights_check["Target Amount MXN"] = weights_check["Target Weight"] * TOTAL_PORTFOLIO_MXN

display(weights_check)
print("Suma de ponderaciones:", round(weights_check["Target Weight"].sum(), 4))
print("Exposición total renta fija:", IPS_WEIGHTS["Mexican government fixed income"] + IPS_WEIGHTS["Domestic corporate and bank fixed income"] + IPS_WEIGHTS["International fixed income"])

,Sleeve,Target Weight,Target Weight %,Target Amount MXN
0,Mexican government fixed income,0.45,45.0,2.250000e+10
1,Domestic corporate and bank fixed income,0.12,12.0,6.000000e+09
2,International fixed income,0.03,3.0,1.500000e+09
3,Mexican equities,0.08,8.0,4.000000e+09
4,International equities,0.15,15.0,7.500000e+09
5,Structured instruments,0.10,10.0,5.000000e+09
6,FIBRAs,0.03,3.0,1.500000e+09
7,Commodities,0.02,2.0,1.000000e+09
8,Cash and liquid assets,0.02,2.0,1.000000e+09


Suma de ponderaciones: 1.0
Exposición total renta fija: 0.6000000000000001


## 3. Lectura robusta del Vector Analítico

In [86]:
# ============================================================
# Lectura robusta del Vector
# ============================================================

VECTOR_PATH = "VectorAnalitico20260416MD.xlsx"

def clean_txt(x):
    x = "" if pd.isna(x) else str(x)
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
    x = x.upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def clean_key(x):
    return re.sub(r"[^A-Z0-9]", "", clean_txt(x))

xls = pd.ExcelFile(VECTOR_PATH)
print("Hojas encontradas:", xls.sheet_names)

sheet_selected = None
header_row = None

for sh in xls.sheet_names:
    preview = pd.read_excel(VECTOR_PATH, sheet_name=sh, header=None, nrows=50)
    for i in range(len(preview)):
        row_clean = [clean_key(v) for v in preview.iloc[i].tolist()]
        if any("ISIN" in v for v in row_clean):
            sheet_selected = sh
            header_row = i
            break
    if sheet_selected is not None:
        break

if sheet_selected is None:
    raise ValueError("No encontré una columna/celda tipo ISIN en las primeras filas del Vector.")

print("Usando hoja:", sheet_selected)
print("Fila de encabezados:", header_row)

vector = pd.read_excel(VECTOR_PATH, sheet_name=sheet_selected, header=header_row)
vector.columns = [str(c).strip() for c in vector.columns]

# Renombrado flexible
rename = {}
for c in vector.columns:
    ck = clean_key(c)
    if "ISIN" in ck:
        rename[c] = "ISIN"
    elif ck in ["TIPOVALOR", "TIPO", "TV", "CLASE", "INSTRUMENTO"]:
        rename[c] = "Tipo"
    elif "EMISORA" in ck or "EMISOR" in ck:
        rename[c] = "Emisora"
    elif "SERIE" in ck:
        rename[c] = "Serie"
    elif "CUPON" in ck:
        rename[c] = "Coupon"
    elif "TASA" in ck or "RENDIMIENTO" in ck or "YIELD" in ck:
        rename[c] = "Yield"
    elif "VENC" in ck or "MATURITY" in ck or "FECHAVTO" in ck or "FECHAVENC" in ck:
        rename[c] = "Maturity"
    elif "DURACION" in ck or "DURATION" in ck:
        rename[c] = "Duration"
    elif "PRECIO" in ck or "PRICE" in ck:
        rename[c] = "Price"
    elif "CALIFIC" in ck or "RATING" in ck:
        rename[c] = "Rating"
    elif "SECTOR" in ck:
        rename[c] = "Sector"

vector = vector.rename(columns=rename)

# Quitar columnas duplicadas después del rename
vector = vector.loc[:, ~vector.columns.duplicated()].copy()

if "ISIN" not in vector.columns:
    raise ValueError(f"No se encontró columna ISIN. Columnas disponibles: {vector.columns.tolist()}")

vector["ISIN"] = vector["ISIN"].astype(str).str.strip().str.upper()
vector = vector[
    vector["ISIN"].notna()
    & (vector["ISIN"] != "")
    & (vector["ISIN"] != "NAN")
].copy()

# Crear columnas necesarias si no existen
for needed_col in ["Tipo", "Emisora", "Serie", "Yield", "Coupon", "Maturity", "Duration", "Price", "Rating", "Sector"]:
    if needed_col not in vector.columns:
        vector[needed_col] = np.nan

# Convertir columnas numéricas de forma segura
for col in ["Yield", "Coupon", "Duration", "Price"]:
    if col in vector.columns:
        if isinstance(vector[col], pd.DataFrame):  # protección extra por duplicados
            vector[col] = vector[col].iloc[:, 0]
        vector[col] = pd.to_numeric(vector[col], errors="coerce")

# Convertir maturity si aplica
vector["Maturity"] = pd.to_datetime(vector["Maturity"], errors="coerce")

print("Columnas finales:")
print(vector.columns.tolist())
print("Filas válidas con ISIN:", len(vector))

display(vector.head())

Hojas encontradas: ['Sheet1']
Usando hoja: Sheet1
Fila de encabezados: 1
Columnas finales:
['FECHA', 'Tipo', 'Emisora', 'Serie', 'Price', 'INTERESES ACUMULADOS', 'Coupon', 'Yield', 'NOMBRE COMPLETO', 'Sector', 'MONTO EMITIDO', 'MONTO EN CIRCULACION', 'FECHA EMISION', 'PLAZO EMISION', 'FECHA VCTO', 'VALOR NOMINAL', 'MONEDA EMISION', 'SUBYACENTE', 'REND. COLOCACION', 'ST COLOCACION', 'FREC. CPN', 'DIAS TRANSC. CPN', 'HECHO DE MKT', 'FECHA U.H.', 'POST COMPRA', 'POST VENTA', 'SPREAD COMPRA', 'SPREAD VENTA', 'MDYS', 'S&P', 'BURSATILIDAD', 'LIQUIDEZ', 'CAMBIO DIARIO', 'CAMBIO SEMANAL', 'SUSPENSION', 'VOLATILIDAD', 'VOLATILIDAD 2', 'Duration', 'CONVEXIDAD', 'VAR', 'DESVIACION STAND', 'VALOR NOMINAL ACTUALIZADO', 'Rating', 'SENSIBILIDAD', 'ISIN', 'Maturity']
Filas válidas con ISIN: 14202


,FECHA,Tipo,Emisora,Serie,Price,INTERESES ACUMULADOS,Coupon,Yield,NOMBRE COMPLETO,Sector,MONTO EMITIDO,MONTO EN CIRCULACION,FECHA EMISION,PLAZO EMISION,FECHA VCTO,VALOR NOMINAL,MONEDA EMISION,SUBYACENTE,REND. COLOCACION,ST COLOCACION,FREC. CPN,DIAS TRANSC. CPN,HECHO DE MKT,FECHA U.H.,POST COMPRA,POST VENTA,SPREAD COMPRA,SPREAD VENTA,MDYS,S&P,BURSATILIDAD,LIQUIDEZ,CAMBIO DIARIO,CAMBIO SEMANAL,SUSPENSION,VOLATILIDAD,VOLATILIDAD 2,Duration,CONVEXIDAD,VAR,DESVIACION STAND,VALOR NOMINAL ACTUALIZADO,Rating,SENSIBILIDAD,ISIN,Maturity
462,20260416,0,CREAL,*,0.000001,0.0,0.0,0.0,CREDITO REAL SAB DE CV SOFOM ER,SERVICIOS FINANCIEROS,-,369.208913,2012-10-17,NaN,NaT,NaN,[MPS] Peso Mexicano (MXN),NaN,NaN,NaN,NaN,NaN,0.354,2022-05-31 00:00:00,-,-,-,-,NaN,NaN,NULA,NaN,0,0,Activo,0,0,NaN,-,0,0,NaN,NaN,-,MX00CR000000,NaT
463,20260416,0,FINDEP,*,8.300000,0.0,0.0,0.0,FINANCIERA INDEPENDENCIA SAB DE CV SOFOM ENR,SERVICIOS FINANCIEROS,-,2801250000,2007-11-01,NaN,NaT,NaN,[MPS] Peso Mexicano (MXN),NaN,NaN,NaN,NaN,NaN,8,2022-09-30 00:00:00,-,-,-,-,NaN,NaN,BAJA,NaN,0,0.0125,Activo,0.025301,0.060929,NaN,-,-0.027645,0.000237,NaN,NaN,-,MX00FI050003,NaT
464,20260416,0,GNP,*,114.000000,0.0,0.0,0.0,GRUPO NACIONAL PROVINCIAL SAB,SERVICIOS FINANCIEROS,-,25549791834,1946-12-03,NaN,NaT,NaN,[MPS] Peso Mexicano (MXN),NaN,NaN,NaN,NaN,NaN,125,2022-09-26 00:00:00,-,-,-,-,NaN,NaN,BAJA,NaN,0,0,Activo,0.003332,0.010446,NaN,-,-0.122012,0.000004,NaN,NaN,-,MXP4960P1070,NaT
465,20260416,0,LASEG,*,2.382416,0.0,0.0,0.0,LA LATINOAMERICANA SEGUROS SA,SERVICIOS FINANCIEROS,-,214417440,1936-08-17,NaN,NaT,NaN,[MPS] Peso Mexicano (MXN),NaN,NaN,NaN,NaN,NaN,1.630608,2011-03-02 00:00:00,-,-,-,-,NaN,NaN,NULA,NaN,0,0,Activo,0.008715,0.020782,NaN,-,0,0.000028,NaN,NaN,-,MXP6171Q1053,NaT
466,20260416,0,UNIFIN,A,1.200000,0.0,0.0,0.0,UNIFIN FINANCIERA SAB DE CV SOFOM ENR,SERVICIOS FINANCIEROS,-,539709572.4,2015-05-22,NaN,NaT,NaN,[MPS] Peso Mexicano (MXN),NaN,NaN,NaN,NaN,NaN,1.8,2022-10-18 00:00:00,-,-,-,-,NaN,NaN,NULA,NaN,0,0,Activo,0,0,NaN,-,-0.053333,0,NaN,NaN,-,MX00UN000002,NaT


## 4. Bonos gubernamentales obligatorios con ponderación interna soberana


In [87]:
# ============================================================
# Bonos gubernamentales obligatorios dentro del 45%
# ============================================================

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def normalize_series(s, higher_is_better=True):
    s = safe_numeric(s)
    if s.notna().sum() == 0:
        return pd.Series(0.5, index=s.index)
    s = s.fillna(s.median())
    min_v, max_v = s.min(), s.max()
    if max_v == min_v:
        out = pd.Series(0.5, index=s.index)
    else:
        out = (s - min_v) / (max_v - min_v)
    return out if higher_is_better else 1 - out

def score_to_weights(score, total_weight, min_w=None, max_w=None):
    """
    Convierte un score financiero en pesos que suman total_weight.
    Usa softmax para evitar pesos extremos. Opcionalmente aplica límites.
    """
    score = safe_numeric(score).fillna(0)
    n = len(score)
    if n == 0:
        return pd.Series(dtype=float)
    if n == 1:
        return pd.Series([total_weight], index=score.index)

    score_std = score.std()
    z = (score - score.mean()) / (score_std if score_std and score_std > 0 else 1)
    raw = np.exp(z)
    weights = pd.Series(raw / raw.sum() * total_weight, index=score.index)

    if min_w is not None or max_w is not None:
        min_w = 0 if min_w is None else min_w
        max_w = total_weight if max_w is None else max_w
        for _ in range(20):
            weights = weights.clip(lower=min_w, upper=max_w)
            diff = total_weight - weights.sum()
            free = (weights > min_w + 1e-12) & (weights < max_w - 1e-12)
            if abs(diff) < 1e-12 or free.sum() == 0:
                break
            weights.loc[free] += diff * (weights.loc[free] / weights.loc[free].sum())
        weights = weights / weights.sum() * total_weight

    return weights

mandatory_gov = pd.DataFrame([
    ["MXBIGO000YV0", "CETES (91)",  "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "BI", "09/07/2026", 0.23, "CETES"],
    ["MXBIGO000YR8", "CETES (182)", "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "BI", "03/09/2026", 0.39, "CETES"],
    ["MXBIGO000YW8", "CETES (364)", "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "BI", "01/04/2027", 0.97, "CETES"],
    ["MXBIGO000YP2", "CETES (721)", "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "BI", "17/02/2028", 1.87, "CETES"],
    ["MXIMBP060209", "IPABONO",     "INSTITUTO PARA LA PROTECCION AL AHORRO BANCARIO", "IM", "01/02/2029", 2.58, "IPAB"],
    ["MXIQBP0701X0", "IPABONO",     "INSTITUTO PARA LA PROTECCION AL AHORRO BANCARIO", "IQ", "16/01/2031", 4.08, "IPAB"],
    ["MXISBP0401R5", "IPABONO",     "INSTITUTO PARA LA PROTECCION AL AHORRO BANCARIO", "IS", "07/04/2033", 5.69, "IPAB"],
    ["MXLDGO000579", "BONDES",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "LD", "06/08/2026", 0.31, "BONDES"],
    ["MXLFGO0003M3", "BONDES",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "LF", "19/04/2035", 6.80, "BONDES"],
    ["MXLGGO0000C8", "BONDES",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "LG", "24/07/2031", 4.49, "BONDES"],
    ["MX0SGO0000P9", "UDIBONOS",    "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "S",  "24/08/2034", 7.21, "UDIBONOS"],
    ["MX0SGO000098", "UDIBONOS",    "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "S",  "15/11/2040", 11.05, "UDIBONOS"],
    ["MX0MGO0000J5", "BONOSM",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "M",  "18/11/2038", 7.51, "BONOS M 10-20 años"],
    ["MXMSGO000001", "BONOSM",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "MS", "24/05/2035", 6.41, "BONOS M 10-20 años"],
    ["MX0MGO000102", "BONOSM",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "M",  "07/11/2047", 9.22, "BONOS M 20-30 años"],
    ["MX0MGO0001E4", "BONOSM",      "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "M",  "31/07/2053", 9.94, "BONOS M 20-30 años"],
    ["US91086QAV05", "UMS",         "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "",   "11/01/2040", np.nan, "UMS"],
    ["JP548400DG68", "UMS",         "SECRETARIA DE HACIENDA Y CREDITO PUBLICO", "",   "16/06/2036", np.nan, "UMS"],
], columns=["ISIN", "Tipo", "Emisora", "Serie", "Maturity", "Duration", "Sovereign Bucket"])

# Ponderaciones por instrumento/bucket soberano: NO MODIFICAR.
sovereign_bucket_weights = {
    "BONOS M 10-20 años": 0.30,
    "BONOS M 20-30 años": 0.15,
    "UDIBONOS": 0.25,
    "BONDES": 0.12,
    "IPAB": 0.08,
    "CETES": 0.05,
    "UMS": 0.05,
}

if round(sum(sovereign_bucket_weights.values()), 10) != 1:
    raise ValueError("Las ponderaciones internas del bucket soberano no suman 100%.")

gov = mandatory_gov.copy()
gov["Maturity"] = pd.to_datetime(gov["Maturity"], dayfirst=True, errors="coerce")

# Enriquecer con datos del vector si existen.
vector_fin_cols = [c for c in ["ISIN", "Yield", "Coupon", "Price", "Rating", "Sector"] if c in vector.columns]
if len(vector_fin_cols) > 1:
    gov = gov.merge(
        vector[vector_fin_cols].drop_duplicates(subset=["ISIN"]),
        on="ISIN",
        how="left"
    )

# Supuestos conservadores si el vector no trae yield/cupón para algún instrumento.
fallback_gov_yields = {
    "CETES": 0.090,
    "IPAB": 0.092,
    "BONDES": 0.091,
    "UDIBONOS": 0.045,              # real yield proxy
    "BONOS M 10-20 años": 0.094,
    "BONOS M 20-30 años": 0.096,
    "UMS": 0.055,
}

gov["Yield Used"] = safe_numeric(gov["Yield"] if "Yield" in gov.columns else np.nan)
gov.loc[gov["Yield Used"] > 1, "Yield Used"] = gov.loc[gov["Yield Used"] > 1, "Yield Used"] / 100
gov["Yield Used"] = gov.apply(
    lambda r: r["Yield Used"] if pd.notna(r["Yield Used"]) else fallback_gov_yields.get(r["Sovereign Bucket"], np.nan),
    axis=1
)

# Score financiero para repartir dentro de cada bucket:
# mayor yield = mejor; menor duración = menor riesgo; vencimiento más cercano = más liquidez.
gov["Yield Score"] = gov.groupby("Sovereign Bucket")["Yield Used"].transform(lambda x: normalize_series(x, True))
gov["Duration Score"] = gov.groupby("Sovereign Bucket")["Duration"].transform(lambda x: normalize_series(x, False))
gov["Maturity Days"] = (gov["Maturity"] - pd.Timestamp.today().normalize()).dt.days
gov["Liquidity Score"] = gov.groupby("Sovereign Bucket")["Maturity Days"].transform(lambda x: normalize_series(x, False))

gov["Financial Score"] = (
    0.55 * gov["Yield Score"] +
    0.35 * gov["Duration Score"] +
    0.10 * gov["Liquidity Score"]
)

gov["Sovereign Bucket Weight"] = gov["Sovereign Bucket"].map(sovereign_bucket_weights)

# Optimiza pesos dentro de cada bucket, pero mantiene fijo el peso total de cada instrumento/bucket.
gov["Weight within Sovereign Bucket"] = 0.0
for bucket, target_weight in sovereign_bucket_weights.items():
    idx = gov.index[gov["Sovereign Bucket"] == bucket]
    if len(idx) == 0:
        continue

    # límites internos del bucket para no concentrar excesivamente
    n = len(idx)
    min_w = 0.15 * target_weight / n if n > 1 else target_weight
    max_w = 0.65 * target_weight if n > 1 else target_weight

    gov.loc[idx, "Weight within Sovereign Bucket"] = score_to_weights(
        gov.loc[idx, "Financial Score"],
        total_weight=target_weight,
        min_w=min_w,
        max_w=max_w
    )

gov["Weight"] = IPS_WEIGHTS["Mexican government fixed income"] * gov["Weight within Sovereign Bucket"]

gov["Sleeve"] = "Mexican government fixed income"
gov["Asset Name"] = gov["Tipo"] + " " + gov["Serie"].fillna("").astype(str)
gov["Source"] = "Mandatory IPS government list; optimized inside fixed sovereign buckets"

# Validación de ponderaciones internas vs. IPS.
sovereign_validation = (
    gov.groupby("Sovereign Bucket", as_index=False)
    .agg(
        Assets=("ISIN", "count"),
        Weight_within_Sovereign_Bucket=("Weight within Sovereign Bucket", "sum"),
        Portfolio_Weight=("Weight", "sum"),
        Weighted_Yield=("Yield Used", lambda x: np.nan),
        Weighted_Duration=("Duration", lambda x: np.nan),
    )
)
for i, row in sovereign_validation.iterrows():
    bucket = row["Sovereign Bucket"]
    temp = gov[gov["Sovereign Bucket"] == bucket].copy()
    w = temp["Weight within Sovereign Bucket"]
    sovereign_validation.loc[i, "Weighted_Yield"] = np.average(temp["Yield Used"].fillna(0), weights=w)
    sovereign_validation.loc[i, "Weighted_Duration"] = np.average(temp["Duration"].fillna(temp["Duration"].median()), weights=w)

sovereign_validation["Target within Sovereign Bucket"] = sovereign_validation["Sovereign Bucket"].map(sovereign_bucket_weights)
sovereign_validation["Difference"] = sovereign_validation["Weight_within_Sovereign_Bucket"] - sovereign_validation["Target within Sovereign Bucket"]

cols_gov = [
    "ISIN", "Tipo", "Emisora", "Serie", "Maturity", "Duration", "Sovereign Bucket",
    "Yield Used", "Financial Score", "Weight within Sovereign Bucket", "Weight",
    "Yield", "Coupon", "Price", "Rating", "Sector", "Sleeve", "Asset Name", "Source"
]
cols_gov = [c for c in cols_gov if c in gov.columns]

display(gov[cols_gov].sort_values(["Sovereign Bucket", "Weight within Sovereign Bucket"], ascending=[True, False]))
display(sovereign_validation)
print("Gobierno - activos:", len(gov))
print("Gobierno - peso soberano total:", round(gov["Weight within Sovereign Bucket"].sum(), 6))
print("Gobierno - peso total portafolio:", round(gov["Weight"].sum(), 6))

,ISIN,Tipo,Emisora,Serie,Maturity,Duration,Sovereign Bucket,Yield Used,Financial Score,Weight within Sovereign Bucket,Weight,Yield,Coupon,Price,Rating,Sector,Sleeve,Asset Name,Source
9,MXLGGO0000C8,BONDES,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,LG,2031-07-24,4.49,BONDES,0.214500,0.626390,0.077032,0.034664,0.214500,6.75,99.040014,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONDES LG,Mandatory IPS government list; optimized insid...
8,MXLFGO0003M3,BONDES,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,LF,2035-04-19,6.80,BONDES,0.243200,0.550000,0.032481,0.014616,0.243200,6.77,98.353658,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONDES LF,Mandatory IPS government list; optimized insid...
7,MXLDGO000579,BONDES,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,LD,2026-08-06,0.31,BONDES,0.070001,0.450000,0.010487,0.004719,0.070001,6.76,99.978399,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONDES LD,Mandatory IPS government list; optimized insid...
13,MXMSGO000001,BONOSM,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,MS,2035-05-24,6.41,BONOS M 10-20 años,0.000000,0.725000,0.195000,0.087750,0.000000,8.00,93.918466,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONOSM MS,Mandatory IPS government list; optimized insid...
12,MX0MGO0000J5,BONOSM,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,M,2038-11-18,7.51,BONOS M 10-20 años,0.000000,0.275000,0.105000,0.047250,0.000000,8.50,94.534159,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONOSM M,Mandatory IPS government list; optimized insid...
14,MX0MGO000102,BONOSM,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,M,2047-11-07,9.22,BONOS M 20-30 años,0.000000,0.725000,0.097500,0.043875,0.000000,8.00,86.428944,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONOSM M,Mandatory IPS government list; optimized insid...
15,MX0MGO0001E4,BONOSM,SECRETARIA DE HACIENDA Y CREDITO PUBLICO,M,2053-07-31,9.94,BONOS M 20-30 años,0.000000,0.275000,0.052500,0.023625,0.000000,8.00,85.658996,AAA(mex),GUBERNAMENTAL,Mexican government fixed income,BONOSM M,Mandatory IPS government list; optimized insid...
0,MXBIGO000YV0,CETES (91),SECRETARIA DE HACIENDA Y CREDITO PUBLICO,BI,2026-07-09,0.23,CETES,0.000000,0.725000,0.021881,0.009847,0.000000,0.00,9.857888,F1+(mex),GUBERNAMENTAL,Mexican government fixed income,CETES (91) BI,Mandatory IPS government list; optimized insid...
1,MXBIGO000YR8,CETES (182),SECRETARIA DE HACIENDA Y CREDITO PUBLICO,BI,2026-09-03,0.39,CETES,0.000000,0.681330,0.017656,0.007945,0.000000,0.00,9.738679,F1+(mex),GUBERNAMENTAL,Mexican government fixed income,CETES (182) BI,Mandatory IPS government list; optimized insid...
2,MXBIGO000YW8,CETES (364),SECRETARIA DE HACIENDA Y CREDITO PUBLICO,BI,2027-04-01,0.97,CETES,0.000000,0.521835,0.008065,0.003629,0.000000,0.00,9.345794,F1+(mex),GUBERNAMENTAL,Mexican government fixed income,CETES (364) BI,Mandatory IPS government list; optimized insid...


,Sovereign Bucket,Assets,Weight_within_Sovereign_Bucket,Portfolio_Weight,Weighted_Yield,Weighted_Duration,Target within Sovereign Bucket,Difference
0,BONDES,3,0.12,0.0540,0.209640,4.749946,0.12,0.000000e+00
1,BONOS M 10-20 años,2,0.30,0.1350,0.000000,6.795000,0.30,0.000000e+00
2,BONOS M 20-30 años,2,0.15,0.0675,0.000000,9.472000,0.15,0.000000e+00
3,CETES,4,0.05,0.0225,0.000000,0.484520,0.05,0.000000e+00
4,IPAB,3,0.08,0.0360,0.237581,4.988190,0.08,1.387779e-17
5,UDIBONOS,2,0.25,0.1125,0.000000,8.554000,0.25,0.000000e+00
6,UMS,2,0.05,0.0225,0.055000,NaN,0.05,0.000000e+00


Gobierno - activos: 18
Gobierno - peso soberano total: 1.0
Gobierno - peso total portafolio: 0.45


In [88]:
# ============================================================
# Gráfico interactivo: ponderación interna del bucket soberano
# Versión sin customdata para evitar errores de hovertemplate
# ============================================================

import plotly.graph_objects as go

sovereign_validation = sovereign_validation.copy()

sovereign_validation["Portfolio_Weight"] = (
    sovereign_validation["Weight_within_Sovereign_Bucket"]
    * IPS_WEIGHTS["Mexican government fixed income"]
)

sovereign_validation["Hover_Text"] = (
    "<b>" + sovereign_validation["Sovereign Bucket"].astype(str) + "</b><br>" +
    "Weight inside sovereign bucket: " + 
    (sovereign_validation["Weight_within_Sovereign_Bucket"] * 100).round(2).astype(str) + "%<br>" +
    "Portfolio weight: " + 
    (sovereign_validation["Portfolio_Weight"] * 100).round(2).astype(str) + "%<br>" +
    "Number of assets: " + sovereign_validation["Assets"].astype(int).astype(str)
)

fig = go.Figure(
    data=[
        go.Pie(
            labels=sovereign_validation["Sovereign Bucket"],
            values=sovereign_validation["Weight_within_Sovereign_Bucket"],
            hole=0.35,
            textinfo="percent+label",
            textposition="inside",
            hoverinfo="text",
            hovertext=sovereign_validation["Hover_Text"]
        )
    ]
)

fig.update_layout(
    title="Sovereign Bucket Allocation inside Mexican Government Fixed Income",
    width=850,
    height=600,
    legend_title_text="Instrument"
)

fig.show()

## 5. Selección de corporativos y bancos nacionales desde el Vector

In [89]:
# ============================================================
# Selección de ~30 instrumentos corporativos/bancarios
# Ponderación NO igual: se optimiza por rendimiento, riesgo, calidad de datos y diversificación.
# ============================================================

N_CORPORATE = 30

gov_keywords = [
    "SECRETARIA DE HACIENDA", "GOBIERNO", "BANCO DE MEXICO", "BANXICO",
    "INSTITUTO PARA LA PROTECCION", "IPAB", "TESORERIA", "ESTADOS UNIDOS MEXICANOS"
]

v = vector.copy()
v["Emisora_clean"] = v["Emisora"].astype(str).str.upper()
v["Tipo_clean"] = v["Tipo"].astype(str).str.upper()

is_gov_emitter = v["Emisora_clean"].apply(lambda x: any(k in x for k in gov_keywords))
is_mandatory_gov = v["ISIN"].isin(mandatory_gov["ISIN"])

corp_candidates = v[~is_gov_emitter & ~is_mandatory_gov].copy()
corp_candidates = corp_candidates.drop_duplicates(subset=["ISIN"]).copy()

# Normalización de variables financieras.
for c in ["Yield", "Coupon", "Duration", "Price"]:
    if c in corp_candidates.columns:
        corp_candidates[c] = safe_numeric(corp_candidates[c])

if "Yield" in corp_candidates.columns:
    corp_candidates.loc[corp_candidates["Yield"] > 1, "Yield"] = corp_candidates.loc[corp_candidates["Yield"] > 1, "Yield"] / 100

# Score de calidad de datos.
corp_candidates["data_score"] = 0
corp_candidates["data_score"] += corp_candidates["Yield"].notna().astype(int) * 3
corp_candidates["data_score"] += corp_candidates["Coupon"].notna().astype(int) * 2
corp_candidates["data_score"] += corp_candidates["Maturity"].notna().astype(int) * 2
corp_candidates["data_score"] += corp_candidates["Duration"].notna().astype(int) * 1
corp_candidates["Price_Sanity"] = corp_candidates["Price"].between(80, 120).astype(int) if "Price" in corp_candidates.columns else 0
corp_candidates["data_score"] += corp_candidates["Price_Sanity"]

# Score financiero para selección:
# - mayor yield/cupón ayuda
# - menor duración reduce sensibilidad a tasas
# - precio razonable y data completa ayudan
corp_candidates["Yield Score"] = normalize_series(corp_candidates["Yield"], True)
corp_candidates["Coupon Score"] = normalize_series(corp_candidates["Coupon"], True)
corp_candidates["Duration Score"] = normalize_series(corp_candidates["Duration"], False)
corp_candidates["Data Quality Score"] = normalize_series(corp_candidates["data_score"], True)

corp_candidates["Selection Score"] = (
    0.45 * corp_candidates["Yield Score"] +
    0.20 * corp_candidates["Coupon Score"] +
    0.20 * corp_candidates["Duration Score"] +
    0.15 * corp_candidates["Data Quality Score"]
)

# Diversificación por emisora: primero toma los mejores por emisor.
corp_candidates = corp_candidates.sort_values(["Selection Score", "Yield"], ascending=[False, False])
corp_selected = corp_candidates.groupby("Emisora", dropna=False).head(2)

# Completar hasta 30 si faltan.
if len(corp_selected) < N_CORPORATE:
    needed = N_CORPORATE - len(corp_selected)
    remaining = corp_candidates[~corp_candidates["ISIN"].isin(corp_selected["ISIN"])]
    corp_selected = pd.concat([corp_selected, remaining.head(needed)], ignore_index=True)

corp_selected = corp_selected.head(N_CORPORATE).copy()
corp_selected["Sleeve"] = "Domestic corporate and bank fixed income"
corp_selected["Asset Name"] = corp_selected["Emisora"].astype(str) + " " + corp_selected["Serie"].astype(str)
corp_selected["Source"] = "Vector Analítico selection; optimized by yield/risk/data quality"

# Ponderación optimizada dentro del sleeve corporativo/bancario.
# No se divide 12% entre 30 en partes iguales.
n_corp = max(len(corp_selected), 1)
min_asset_weight = IPS_WEIGHTS["Domestic corporate and bank fixed income"] * 0.35 / n_corp
max_asset_weight = IPS_WEIGHTS["Domestic corporate and bank fixed income"] * 0.075  # máx. 0.90% del portafolio por bono

corp_selected["Weight"] = score_to_weights(
    corp_selected["Selection Score"],
    total_weight=IPS_WEIGHTS["Domestic corporate and bank fixed income"],
    min_w=min_asset_weight,
    max_w=max_asset_weight
).values

# Control adicional de concentración por emisora: si algún emisor supera 15% del sleeve, redistribuir el exceso.
issuer_cap = IPS_WEIGHTS["Domestic corporate and bank fixed income"] * 0.15
for _ in range(10):
    issuer_weights = corp_selected.groupby("Emisora")["Weight"].sum()
    over_issuers = issuer_weights[issuer_weights > issuer_cap]
    if over_issuers.empty:
        break
    excess_total = 0
    for issuer, total_w in over_issuers.items():
        mask = corp_selected["Emisora"] == issuer
        scale = issuer_cap / total_w
        old = corp_selected.loc[mask, "Weight"].copy()
        corp_selected.loc[mask, "Weight"] = old * scale
        excess_total += (old - corp_selected.loc[mask, "Weight"]).sum()
    under_mask = ~corp_selected["Emisora"].isin(over_issuers.index)
    if under_mask.sum() > 0 and excess_total > 0:
        corp_selected.loc[under_mask, "Weight"] += (
            corp_selected.loc[under_mask, "Weight"] / corp_selected.loc[under_mask, "Weight"].sum()
        ) * excess_total

corp_selected["Weight"] = corp_selected["Weight"] / corp_selected["Weight"].sum() * IPS_WEIGHTS["Domestic corporate and bank fixed income"]

cols_show = [
    "ISIN", "Asset Name", "Tipo", "Emisora", "Serie", "Yield", "Coupon", "Maturity",
    "Duration", "Price", "Rating", "Sector", "Selection Score", "Weight"
]
cols_show = [c for c in cols_show if c in corp_selected.columns]

display(corp_selected[cols_show].sort_values("Weight", ascending=False))
print("Corporativos seleccionados:", len(corp_selected))
print("Peso corporativo total:", round(corp_selected["Weight"].sum(), 6))
print("Peso máximo por activo:", round(corp_selected["Weight"].max(), 6))
print("Peso mínimo por activo:", round(corp_selected["Weight"].min(), 6))

,ISIN,Asset Name,Tipo,Emisora,Serie,Yield,Coupon,Maturity,Duration,Price,Rating,Sector,Selection Score,Weight
12792,MX97BR040092,BRHCCB 08-5U,97,BRHCCB,08-5U,5.985362,6.55,NaT,NaN,35.979549,RETIRADA,SERVICIOS FINANCIEROS,0.734627,0.010506
11679,MX90GC040026,GCDMXCB 18V,90,GCDMXCB,18V,0.995000,9.93,NaT,2.182583,104.098188,-,SERVICIOS FINANCIEROS,0.701948,0.010506
12701,MX95CF0500M1,CFE 22-2S,95,CFE,22-2S,0.808868,10.82,NaT,3.483811,106.777475,AAA(mex),SERVICIOS PÚBLICOS,0.697439,0.010506
11826,MX91EN090011,ENGENCB 24-2,91,ENGENCB,24-2,0.020195,12.98,NaT,2.982427,109.348493,AAA(mex),SERVICIOS FINANCIEROS,0.696119,0.010506
12659,MX94CO0H00D7,COMPART 21-2S,94,COMPART,21-2S,0.802300,9.19,NaT,0.553362,100.786933,AA(mex),SERVICIOS FINANCIEROS,0.693736,0.010506
13023,MX0FBA062656,BANOBRA 23432,F,BANOBRA,23432,0.180000,10.83,NaT,0.375665,101.412018,AAA(mex),GUBERNAMENTAL,0.690297,0.010506
11932,MX91IN4J0078,INVEX 22,91,INVEX,22,0.958987,8.25,NaT,0.191927,100.047808,A+(mex),SERVICIOS FINANCIEROS,0.689827,0.010506
12118,MX91VW0100M4,VWLEASE 24-2,91,VWLEASE,24-2,0.968545,11.03,NaT,2.173560,105.319592,-,SERVICIOS FINANCIEROS,0.716530,0.010506
11979,MX91ME090046,MEGA 24-2X,91,MEGA,24-2X,0.800000,10.00,NaT,3.864035,102.971976,AAA(mex),SERVICIOS DE TELECOMUNICACIONES,0.682745,0.001634
12082,MX91TO0000G9,TOYOTA 24-2,91,TOYOTA,24-2,0.288626,10.66,NaT,1.784505,104.869603,AAA(mex),SERVICIOS FINANCIEROS,0.683212,0.001634


Corporativos seleccionados: 30
Peso corporativo total: 0.12
Peso máximo por activo: 0.010506
Peso mínimo por activo: 0.001634


## 6. Activos proxy para otros sleeves

In [90]:
# ============================================================
# Activos líquidos/proxy para sleeves fuera del Vector local
# Ponderación NO igual: se optimiza por rendimiento/riesgo histórico cuando Yahoo descarga datos.
# Si Yahoo falla, se usa un score financiero estratégico de respaldo.
# ============================================================

other_assets = pd.DataFrame([
    # Sleeve, Asset Name, ISIN, Yahoo proxy
    ["International fixed income", "iShares Core U.S. Aggregate Bond ETF", "US4642872265", "AGG"],
    ["International fixed income", "iShares J.P. Morgan USD Emerging Markets Bond ETF", "US4642882819", "EMB"],

    ["Mexican equities", "iShares MSCI Mexico ETF", "US4642868222", "EWW"],
    ["Mexican equities", "Mexico Equity Proxy / IPC", "", "^MXX"],

    ["International equities", "SPDR S&P 500 ETF Trust", "US78462F1030", "SPY"],
    ["International equities", "iShares MSCI ACWI ETF", "US4642882579", "ACWI"],
    ["International equities", "Invesco QQQ Trust", "US46090E1038", "QQQ"],

    ["Structured instruments", "Global multi-asset structured proxy", "", "AOR"],
    ["Structured instruments", "NASDAQ structured payoff proxy", "US46090E1038", "QQQ"],

    ["FIBRAs", "FIBRA Uno", "MXCFFU000001", "FUNO11.MX"],
    ["FIBRAs", "FIBRA Monterrey", "MXCFFM010003", "FMTY14.MX"],

    ["Commodities", "SPDR Gold Shares", "US78463V1070", "GLD"],
    ["Commodities", "iShares Silver Trust", "US46428Q1094", "SLV"],
    ["Commodities", "United States Oil Fund", "US91232N2071", "USO"],

    ["Cash and liquid assets", "iShares Short Treasury Bond ETF", "US4642886794", "SHV"],
    ["Cash and liquid assets", "SPDR Bloomberg 1-3 Month T-Bill ETF", "US78468R6633", "BIL"],
], columns=["Sleeve", "Asset Name", "ISIN", "Yahoo Proxy"])

# Pesos estratégicos de respaldo si Yahoo no descarga datos suficientes.
fallback_proxy_score = {
    "AGG": 0.55, "EMB": 0.45,
    "EWW": 0.60, "^MXX": 0.40,
    "SPY": 0.45, "ACWI": 0.35, "QQQ": 0.20,
    "AOR": 0.55,
    "FUNO11.MX": 0.55, "FMTY14.MX": 0.45,
    "GLD": 0.50, "SLV": 0.25, "USO": 0.25,
    "SHV": 0.70, "BIL": 0.30,
}

def download_proxy_prices(tickers, period="3y"):
    try:
        data = yf.download(
            tickers,
            period=period,
            auto_adjust=True,
            progress=False,
            group_by="column",
            threads=True
        )
        if data is None or len(data) == 0:
            return pd.DataFrame()
        if isinstance(data.columns, pd.MultiIndex):
            if "Close" in data.columns.get_level_values(0):
                prices = data["Close"].copy()
            else:
                prices = data.xs("Close", axis=1, level=1, drop_level=True).copy()
        else:
            prices = data[["Close"]].rename(columns={"Close": tickers[0]})
        return prices.dropna(how="all")
    except Exception as e:
        print("No se pudieron descargar precios preliminares de Yahoo:", e)
        return pd.DataFrame()

tickers_other = other_assets["Yahoo Proxy"].dropna().unique().tolist()
proxy_prices_pre = download_proxy_prices(tickers_other, period="3y")
proxy_returns_pre = proxy_prices_pre.pct_change().dropna(how="all") if not proxy_prices_pre.empty else pd.DataFrame()

proxy_perf = []
for ticker in tickers_other:
    if ticker in proxy_returns_pre.columns and proxy_returns_pre[ticker].dropna().shape[0] > 60:
        r = proxy_returns_pre[ticker].dropna()
        ann_ret = (1 + r).prod() ** (252 / len(r)) - 1
        ann_vol = r.std() * np.sqrt(252)
        sharpe_like = ann_ret / ann_vol if ann_vol and ann_vol > 0 else np.nan
        proxy_perf.append([ticker, ann_ret, ann_vol, sharpe_like])
    else:
        proxy_perf.append([ticker, np.nan, np.nan, np.nan])

proxy_perf = pd.DataFrame(proxy_perf, columns=["Yahoo Proxy", "Proxy Annual Return", "Proxy Annual Volatility", "Proxy Sharpe Like"])
other_assets = other_assets.merge(proxy_perf, on="Yahoo Proxy", how="left")

other_assets["Fallback Score"] = other_assets["Yahoo Proxy"].map(fallback_proxy_score).fillna(0.50)

# Score por proxy:
# - si hay Yahoo suficiente: Sharpe/rendimiento favorecen; volatilidad penaliza.
# - si no hay Yahoo: usa score estratégico de respaldo.
other_assets["Proxy Return Score"] = other_assets.groupby("Sleeve")["Proxy Annual Return"].transform(lambda x: normalize_series(x, True))
other_assets["Proxy Risk Score"] = other_assets.groupby("Sleeve")["Proxy Annual Volatility"].transform(lambda x: normalize_series(x, False))
other_assets["Proxy Sharpe Score"] = other_assets.groupby("Sleeve")["Proxy Sharpe Like"].transform(lambda x: normalize_series(x, True))
other_assets["Fallback Score Norm"] = other_assets.groupby("Sleeve")["Fallback Score"].transform(lambda x: normalize_series(x, True))

has_yahoo = other_assets["Proxy Sharpe Like"].notna()
other_assets["Optimization Score"] = np.where(
    has_yahoo,
    0.50 * other_assets["Proxy Sharpe Score"] + 0.30 * other_assets["Proxy Return Score"] + 0.20 * other_assets["Proxy Risk Score"],
    other_assets["Fallback Score Norm"]
)

# Optimizar dentro de cada sleeve sin cambiar el peso total IPS del sleeve.
other_assets["Weight"] = 0.0
other_assets["Sleeve Internal Weight"] = 0.0

for sleeve, target_w in IPS_WEIGHTS.items():
    idx = other_assets.index[other_assets["Sleeve"] == sleeve]
    if len(idx) == 0:
        continue
    n = len(idx)
    min_w = target_w * 0.20 / n if n > 1 else target_w
    max_w = target_w * 0.75 if n > 1 else target_w
    w = score_to_weights(
        other_assets.loc[idx, "Optimization Score"],
        total_weight=target_w,
        min_w=min_w,
        max_w=max_w
    )
    other_assets.loc[idx, "Weight"] = w.values
    other_assets.loc[idx, "Sleeve Internal Weight"] = other_assets.loc[idx, "Weight"] / target_w

other_assets["Tipo"] = "ETF / Proxy"
other_assets["Emisora"] = other_assets["Asset Name"]
other_assets["Serie"] = ""
other_assets["Yield"] = np.nan
other_assets["Coupon"] = np.nan
other_assets["Maturity"] = pd.NaT
other_assets["Duration"] = np.nan
other_assets["Price"] = np.nan
other_assets["Rating"] = np.nan
other_assets["Sector"] = np.nan
other_assets["Source"] = "Yahoo Finance proxy; optimized by historical return/risk when available"

display(other_assets[[
    "Sleeve", "Asset Name", "ISIN", "Yahoo Proxy", "Proxy Annual Return",
    "Proxy Annual Volatility", "Proxy Sharpe Like", "Optimization Score",
    "Sleeve Internal Weight", "Weight"
]].sort_values(["Sleeve", "Weight"], ascending=[True, False]))

print("Peso otros sleeves:", round(other_assets["Weight"].sum(), 6))

,Sleeve,Asset Name,ISIN,Yahoo Proxy,Proxy Annual Return,Proxy Annual Volatility,Proxy Sharpe Like,Optimization Score,Sleeve Internal Weight,Weight
15,Cash and liquid assets,SPDR Bloomberg 1-3 Month T-Bill ETF,US78468R6633,BIL,0.045579,0.002399,19.000100,0.700000,0.750000,0.015000
14,Cash and liquid assets,iShares Short Treasury Bond ETF,US4642886794,SHV,0.045623,0.002474,18.437907,0.300000,0.250000,0.005000
11,Commodities,SPDR Gold Shares,US78463V1070,GLD,0.317487,0.194787,1.629921,0.824352,0.643145,0.012863
12,Commodities,iShares Silver Trust,US46428Q1094,SLV,0.424212,0.396319,1.070379,0.491414,0.269345,0.005387
13,Commodities,United States Oil Fund,US91232N2071,USO,0.241930,0.334482,0.723297,0.061367,0.087510,0.001750
10,FIBRAs,FIBRA Monterrey,MXCFFM010003,FMTY14.MX,0.169584,0.229624,0.738530,0.700000,0.750000,0.022500
9,FIBRAs,FIBRA Uno,MXCFFU000001,FUNO11.MX,0.172005,0.274907,0.625682,0.300000,0.250000,0.007500
6,International equities,Invesco QQQ Trust,US46090E1038,QQQ,0.272910,0.194536,1.402875,0.800000,0.649892,0.097484
4,International equities,SPDR S&P 500 ETF Trust,US78462F1030,SPY,0.207432,0.150242,1.380653,0.526999,0.261917,0.039288
5,International equities,iShares MSCI ACWI ETF,US4642882579,ACWI,0.190172,0.141108,1.347704,0.200000,0.088191,0.013229


Peso otros sleeves: 0.43


## 7. Cartera completa de activos seleccionados

In [91]:
# ============================================================
# Consolidación de cartera
# ============================================================

gov_assets = gov.copy()
gov_assets["Yahoo Proxy"] = np.nan

corp_assets = corp_selected.copy()
corp_assets["Yahoo Proxy"] = np.nan

common_cols = [
    "Sleeve", "Asset Name", "ISIN", "Tipo", "Emisora", "Serie",
    "Yield", "Yield Used", "Coupon", "Maturity", "Duration", "Price", "Rating", "Sector",
    "Financial Score", "Selection Score", "Optimization Score",
    "Weight", "Yahoo Proxy", "Source"
]

for df_name in ["gov_assets", "corp_assets", "other_assets"]:
    df = globals()[df_name]
    for c in common_cols:
        if c not in df.columns:
            df[c] = np.nan
    globals()[df_name] = df[common_cols].copy()

portfolio_assets = pd.concat([gov_assets, corp_assets, other_assets], ignore_index=True)

portfolio_assets["Weight"] = safe_numeric(portfolio_assets["Weight"]).fillna(0)
portfolio_assets["Weight %"] = portfolio_assets["Weight"] * 100
portfolio_assets["Amount MXN"] = portfolio_assets["Weight"] * TOTAL_PORTFOLIO_MXN

# Fallback yield assumptions para instrumentos sin yield del vector
fallback_yields = {
    "Mexican government fixed income": 0.095,
    "Domestic corporate and bank fixed income": 0.105,
    "International fixed income": 0.045,
    "Mexican equities": 0.085,
    "International equities": 0.075,
    "Structured instruments": 0.080,
    "FIBRAs": 0.075,
    "Commodities": 0.030,
    "Cash and liquid assets": 0.050,
}

portfolio_assets["Yield Used"] = portfolio_assets.apply(
    lambda r: r["Yield Used"] if pd.notna(r["Yield Used"])
    else (r["Yield"] if pd.notna(r["Yield"]) else fallback_yields.get(r["Sleeve"], np.nan)),
    axis=1
)

# Si el yield viene en porcentaje tipo 9.5 en vez de 0.095, normalizar.
portfolio_assets["Yield Used"] = safe_numeric(portfolio_assets["Yield Used"])
portfolio_assets.loc[portfolio_assets["Yield Used"] > 1, "Yield Used"] = portfolio_assets.loc[portfolio_assets["Yield Used"] > 1, "Yield Used"] / 100

allocation_rows = []
for sleeve, temp in portfolio_assets.groupby("Sleeve"):
    w = temp["Weight"].fillna(0)
    total_w = w.sum()
    allocation_rows.append({
        "Sleeve": sleeve,
        "Weight": total_w,
        "Amount_MXN": temp["Amount MXN"].sum(),
        "Assets": len(temp),
        "Weighted_Yield": np.average(temp["Yield Used"].fillna(0), weights=w) if total_w > 0 else np.nan,
        "Weighted_Duration": np.average(temp["Duration"].fillna(temp["Duration"].median()), weights=w) if total_w > 0 and temp["Duration"].notna().any() else np.nan,
    })

allocation_check = pd.DataFrame(allocation_rows)
allocation_check["Weight %"] = allocation_check["Weight"] * 100
allocation_check["Target Weight"] = allocation_check["Sleeve"].map(IPS_WEIGHTS)
allocation_check["Difference"] = allocation_check["Weight"] - allocation_check["Target Weight"]

portfolio_assets = portfolio_assets.sort_values(["Sleeve", "Weight"], ascending=[True, False]).reset_index(drop=True)

display(allocation_check.sort_values("Sleeve"))
display(portfolio_assets[[
    "Sleeve", "Asset Name", "ISIN", "Tipo", "Yield Used", "Duration",
    "Weight %", "Amount MXN", "Yahoo Proxy", "Source"
]])
print("Peso total cartera:", round(portfolio_assets["Weight"].sum(), 6))
print("Número total de activos:", len(portfolio_assets))

,Sleeve,Weight,Amount_MXN,Assets,Weighted_Yield,Weighted_Duration,Weight %,Target Weight,Difference
0,Cash and liquid assets,0.02,1.000000e+09,2,0.050000,NaN,2.0,0.02,3.469447e-18
1,Commodities,0.02,1.000000e+09,3,0.030000,NaN,2.0,0.02,0.000000e+00
2,Domestic corporate and bank fixed income,0.12,6.000000e+09,30,0.608856,1.722189,12.0,0.12,0.000000e+00
3,FIBRAs,0.03,1.500000e+09,2,0.075000,NaN,3.0,0.03,0.000000e+00
4,International equities,0.15,7.500000e+09,3,0.075000,NaN,15.0,0.15,2.775558e-17
5,International fixed income,0.03,1.500000e+09,2,0.045000,NaN,3.0,0.03,0.000000e+00
6,Mexican equities,0.08,4.000000e+09,2,0.085000,NaN,8.0,0.08,0.000000e+00
7,Mexican government fixed income,0.45,2.250000e+10,18,0.046913,6.845575,45.0,0.45,0.000000e+00
8,Structured instruments,0.10,5.000000e+09,2,0.080000,NaN,10.0,0.10,0.000000e+00


,Sleeve,Asset Name,ISIN,Tipo,Yield Used,Duration,Weight %,Amount MXN,Yahoo Proxy,Source
0,Cash and liquid assets,SPDR Bloomberg 1-3 Month T-Bill ETF,US78468R6633,ETF / Proxy,0.050000,NaN,1.500000,7.500000e+08,BIL,Yahoo Finance proxy; optimized by historical r...
1,Cash and liquid assets,iShares Short Treasury Bond ETF,US4642886794,ETF / Proxy,0.050000,NaN,0.500000,2.500000e+08,SHV,Yahoo Finance proxy; optimized by historical r...
2,Commodities,SPDR Gold Shares,US78463V1070,ETF / Proxy,0.030000,NaN,1.286290,6.431452e+08,GLD,Yahoo Finance proxy; optimized by historical r...
3,Commodities,iShares Silver Trust,US46428Q1094,ETF / Proxy,0.030000,NaN,0.538690,2.693452e+08,SLV,Yahoo Finance proxy; optimized by historical r...
4,Commodities,United States Oil Fund,US91232N2071,ETF / Proxy,0.030000,NaN,0.175019,8.750960e+07,USO,Yahoo Finance proxy; optimized by historical r...
5,Domestic corporate and bank fixed income,BRHCCB 08-5U,MX97BR040092,97,0.059854,NaN,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
6,Domestic corporate and bank fixed income,VWLEASE 24-2,MX91VW0100M4,91,0.968545,2.173560,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
7,Domestic corporate and bank fixed income,GCDMXCB 18V,MX90GC040026,90,0.995000,2.182583,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
8,Domestic corporate and bank fixed income,CFE 22-2S,MX95CF0500M1,95,0.808868,3.483811,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
9,Domestic corporate and bank fixed income,ENGENCB 24-2,MX91EN090011,91,0.020195,2.982427,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...


Peso total cartera: 1.0
Número total de activos: 64


## 8. Tabla de proxies de Yahoo Finance

In [92]:
# ============================================================
# Tabla de proxies por sleeve
# Para bonos sin ticker directo se usa un proxy de mercado.
# ============================================================

sleeve_proxy_candidates = {
    "Mexican government fixed income": ["CETETRC.MX", "M10TRACISHRS.MX", "UDITRACISHRS.MX", "SHV"],
    "Domestic corporate and bank fixed income": ["CORPTRCISHRS.MX", "LQD", "AGG"],
    "International fixed income": ["AGG", "BND", "EMB"],
    "Mexican equities": ["EWW", "^MXX", "MEXTRAC09.MX"],
    "International equities": ["ACWI", "SPY", "QQQ"],
    "Structured instruments": ["AOR", "QQQ", "SPY"],
    "FIBRAs": ["FUNO11.MX", "FMTY14.MX", "VNQ"],
    "Commodities": ["GLD", "SLV", "USO"],
    "Cash and liquid assets": ["SHV", "BIL", "SGOV"],
}

proxy_explanation = {
    "Mexican government fixed income": "Proxy for Mexican sovereign fixed income due to limited direct Yahoo tickers for individual CETES/BONOS/UDIBONOS/IPABONOS.",
    "Domestic corporate and bank fixed income": "Proxy for domestic corporate/bank debt; fallback uses investment-grade credit ETFs when local tickers are unavailable.",
    "International fixed income": "Proxy for global/U.S. aggregate and emerging-market USD bonds.",
    "Mexican equities": "Proxy for Mexican equity market exposure.",
    "International equities": "Proxy for broad developed/global equity exposure.",
    "Structured instruments": "Proxy for multi-asset/option-like structured payoff exposure using liquid ETFs.",
    "FIBRAs": "Proxy for Mexican real estate trust exposure.",
    "Commodities": "Proxy for liquid commodity exposures.",
    "Cash and liquid assets": "Proxy for short-duration T-bills/cash equivalents.",
}

proxy_table = pd.DataFrame([
    {
        "Sleeve": sleeve,
        "Primary Proxy": candidates[0],
        "Alternative Proxies": ", ".join(candidates[1:]),
        "Explanation": proxy_explanation.get(sleeve, "")
    }
    for sleeve, candidates in sleeve_proxy_candidates.items()
])

display(proxy_table)

,Sleeve,Primary Proxy,Alternative Proxies,Explanation
0,Mexican government fixed income,CETETRC.MX,"M10TRACISHRS.MX, UDITRACISHRS.MX, SHV",Proxy for Mexican sovereign fixed income due t...
1,Domestic corporate and bank fixed income,CORPTRCISHRS.MX,"LQD, AGG",Proxy for domestic corporate/bank debt; fallba...
2,International fixed income,AGG,"BND, EMB",Proxy for global/U.S. aggregate and emerging-m...
3,Mexican equities,EWW,"^MXX, MEXTRAC09.MX",Proxy for Mexican equity market exposure.
4,International equities,ACWI,"SPY, QQQ",Proxy for broad developed/global equity exposure.
5,Structured instruments,AOR,"QQQ, SPY",Proxy for multi-asset/option-like structured p...
6,FIBRAs,FUNO11.MX,"FMTY14.MX, VNQ",Proxy for Mexican real estate trust exposure.
7,Commodities,GLD,"SLV, USO",Proxy for liquid commodity exposures.
8,Cash and liquid assets,SHV,"BIL, SGOV",Proxy for short-duration T-bills/cash equivale...


## 9. Backtesting con Yahoo Finance

In [93]:
# ============================================================
# Descarga de precios y elección automática de proxy disponible
# ============================================================

BACKTEST_START = "2019-01-01"
BACKTEST_END = None

all_candidate_tickers = sorted(set(sum(sleeve_proxy_candidates.values(), [])))
print("Intentando descargar tickers:", all_candidate_tickers)

raw = yf.download(
    tickers=all_candidate_tickers,
    start=BACKTEST_START,
    end=BACKTEST_END,
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True
)

def extract_close_prices(raw_data):
    if isinstance(raw_data.columns, pd.MultiIndex):
        if "Close" in raw_data.columns.get_level_values(0):
            close = raw_data["Close"].copy()
        elif "Adj Close" in raw_data.columns.get_level_values(0):
            close = raw_data["Adj Close"].copy()
        else:
            # último recurso: tomar primer nivel numérico que exista
            close = raw_data.xs(raw_data.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw_data[["Close"]].copy() if "Close" in raw_data.columns else raw_data.copy()
    return close

prices_all = extract_close_prices(raw)
prices_all = prices_all.dropna(axis=1, how="all")
prices_all = prices_all.ffill().dropna(how="all")

available_tickers = list(prices_all.columns)
print("Tickers disponibles:", available_tickers)

selected_sleeve_proxy = {}
for sleeve, candidates in sleeve_proxy_candidates.items():
    selected = None
    for t in candidates:
        if t in prices_all.columns and prices_all[t].dropna().shape[0] > 60:
            selected = t
            break
    selected_sleeve_proxy[sleeve] = selected

selected_proxy_table = pd.DataFrame([
    {"Sleeve": s, "Selected Proxy": p, "Status": "OK" if p else "NO DATA"}
    for s, p in selected_sleeve_proxy.items()
])

display(selected_proxy_table)

missing = selected_proxy_table[selected_proxy_table["Status"] == "NO DATA"]
if len(missing) > 0:
    print("OJO: Algunos sleeves no tuvieron proxy disponible. Revisa conexión/tickers:")
    display(missing)

Intentando descargar tickers: ['ACWI', 'AGG', 'AOR', 'BIL', 'BND', 'CETETRC.MX', 'CORPTRCISHRS.MX', 'EMB', 'EWW', 'FMTY14.MX', 'FUNO11.MX', 'GLD', 'LQD', 'M10TRACISHRS.MX', 'MEXTRAC09.MX', 'QQQ', 'SGOV', 'SHV', 'SLV', 'SPY', 'UDITRACISHRS.MX', 'USO', 'VNQ', '^MXX']
Tickers disponibles: ['ACWI', 'AGG', 'AOR', 'BIL', 'BND', 'CETETRC.MX', 'CORPTRCISHRS.MX', 'EMB', 'EWW', 'FMTY14.MX', 'FUNO11.MX', 'GLD', 'LQD', 'M10TRACISHRS.MX', 'MEXTRAC09.MX', 'QQQ', 'SGOV', 'SHV', 'SLV', 'SPY', 'UDITRACISHRS.MX', 'USO', 'VNQ', '^MXX']


,Sleeve,Selected Proxy,Status
0,Mexican government fixed income,CETETRC.MX,OK
1,Domestic corporate and bank fixed income,CORPTRCISHRS.MX,OK
2,International fixed income,AGG,OK
3,Mexican equities,EWW,OK
4,International equities,ACWI,OK
5,Structured instruments,AOR,OK
6,FIBRAs,FUNO11.MX,OK
7,Commodities,GLD,OK
8,Cash and liquid assets,SHV,OK


In [94]:
# ============================================================
# Construcción de series de retorno por sleeve y cartera total
# ============================================================

sleeve_returns = pd.DataFrame(index=prices_all.index)

for sleeve, ticker in selected_sleeve_proxy.items():
    if ticker is not None and ticker in prices_all.columns:
        sleeve_returns[sleeve] = prices_all[ticker].pct_change()

sleeve_returns = sleeve_returns.dropna(how="all").fillna(0)

# Ponderaciones por sleeve
sleeve_weights = pd.Series(IPS_WEIGHTS)
sleeve_weights = sleeve_weights[sleeve_weights.index.isin(sleeve_returns.columns)]
sleeve_weights = sleeve_weights / sleeve_weights.sum()

portfolio_returns = sleeve_returns.mul(sleeve_weights, axis=1).sum(axis=1)
portfolio_cum = (1 + portfolio_returns).cumprod()

benchmark_candidates = ["ACWI", "SPY", "EWW", "^MXX"]
benchmark_ticker = next((t for t in benchmark_candidates if t in prices_all.columns), None)

if benchmark_ticker:
    benchmark_returns = prices_all[benchmark_ticker].pct_change().reindex(portfolio_returns.index).fillna(0)
    benchmark_cum = (1 + benchmark_returns).cumprod()
else:
    benchmark_returns = pd.Series(index=portfolio_returns.index, data=np.nan)
    benchmark_cum = pd.Series(index=portfolio_returns.index, data=np.nan)

backtesting_table = pd.DataFrame({
    "Date": portfolio_returns.index,
    "Portfolio Daily Return": portfolio_returns.values,
    "Portfolio Cumulative Return Index": portfolio_cum.values,
    "Benchmark Ticker": benchmark_ticker,
    "Benchmark Daily Return": benchmark_returns.values,
    "Benchmark Cumulative Return Index": benchmark_cum.values,
}).set_index("Date")

display(backtesting_table.tail())
print("Benchmark usado:", benchmark_ticker)

,Portfolio Daily Return,Portfolio Cumulative Return Index,Benchmark Ticker,Benchmark Daily Return,Benchmark Cumulative Return Index
Date,,,,,
2026-04-17,0.004195,1.741709,ACWI,0.013432,2.674890
2026-04-20,0.000024,1.741752,ACWI,-0.003247,2.666205
2026-04-21,-0.005916,1.731447,ACWI,-0.011103,2.636602
2026-04-22,0.002658,1.736049,ACWI,0.009614,2.661950
2026-04-23,-0.002271,1.732106,ACWI,-0.006126,2.645642


Benchmark usado: ACWI


## 10. Métricas del portafolio

In [95]:
# ============================================================
# Métricas principales
# ============================================================

TRADING_DAYS = 252
RISK_FREE_RATE = 0.05  # supuesto anual para MXN/cash

def annualized_return(daily_returns):
    daily_returns = daily_returns.dropna()
    if len(daily_returns) == 0:
        return np.nan
    return (1 + daily_returns).prod() ** (TRADING_DAYS / len(daily_returns)) - 1

def annualized_volatility(daily_returns):
    return daily_returns.dropna().std() * np.sqrt(TRADING_DAYS)

def historical_var(daily_returns, confidence=0.95):
    daily_returns = daily_returns.dropna()
    if len(daily_returns) == 0:
        return np.nan
    return -np.percentile(daily_returns, (1 - confidence) * 100)

def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    drawdown = cum_returns / running_max - 1
    return drawdown.min()

port_ann_return = annualized_return(portfolio_returns)
port_ann_vol = annualized_volatility(portfolio_returns)
port_var_95_daily = historical_var(portfolio_returns, 0.95)
port_var_95_annual = port_var_95_daily * np.sqrt(TRADING_DAYS)
port_sharpe = (port_ann_return - RISK_FREE_RATE) / port_ann_vol if port_ann_vol and port_ann_vol != 0 else np.nan
port_yield = np.average(portfolio_assets["Yield Used"].fillna(0), weights=portfolio_assets["Weight"])
port_mdd = max_drawdown(portfolio_cum)

# Alpha y beta contra benchmark
if benchmark_ticker and benchmark_returns.notna().sum() > 60:
    aligned = pd.concat([portfolio_returns, benchmark_returns], axis=1).dropna()
    aligned.columns = ["portfolio", "benchmark"]
    cov = np.cov(aligned["portfolio"], aligned["benchmark"])[0, 1]
    var_b = np.var(aligned["benchmark"])
    beta = cov / var_b if var_b != 0 else np.nan
    alpha_daily = aligned["portfolio"].mean() - beta * aligned["benchmark"].mean()
    alpha_annual = alpha_daily * TRADING_DAYS
else:
    beta = np.nan
    alpha_annual = np.nan

metrics = pd.DataFrame({
    "Metric": [
        "Expected / Backtested Annual Return",
        "Weighted Portfolio Yield",
        "Annualized Volatility",
        "Daily Historical VaR 95%",
        "Annualized Historical VaR 95%",
        "Sharpe Ratio",
        "Alpha vs Benchmark",
        "Beta vs Benchmark",
        "Maximum Drawdown",
        "Benchmark"
    ],
    "Value": [
        port_ann_return,
        port_yield,
        port_ann_vol,
        port_var_95_daily,
        port_var_95_annual,
        port_sharpe,
        alpha_annual,
        beta,
        port_mdd,
        benchmark_ticker
    ]
})

display(metrics)

# VaR monetario
var_mxn_daily = port_var_95_daily * TOTAL_PORTFOLIO_MXN
var_mxn_annual = port_var_95_annual * TOTAL_PORTFOLIO_MXN

print(f"VaR diario 95% estimado: ${var_mxn_daily:,.0f} MXN")
print(f"VaR anualizado 95% estimado: ${var_mxn_annual:,.0f} MXN")

,Metric,Value
0,Expected / Backtested Annual Return,0.076078
1,Weighted Portfolio Yield,0.125424
2,Annualized Volatility,0.061559
3,Daily Historical VaR 95%,0.005395
4,Annualized Historical VaR 95%,0.085649
5,Sharpe Ratio,0.423619
6,Alpha vs Benchmark,0.029661
7,Beta vs Benchmark,0.310552
8,Maximum Drawdown,-0.138824
9,Benchmark,ACWI


VaR diario 95% estimado: $269,769,757 MXN
VaR anualizado 95% estimado: $4,282,462,137 MXN


## 11. Gráficas interactivas con Plotly

In [96]:
# ============================================================
# Gráfico pastel interactivo: ponderación por tipo de activo
# ============================================================

import plotly.graph_objects as go

pie_data = (
    portfolio_assets
    .groupby("Sleeve", as_index=False)
    .agg(
        Weight=("Weight", "sum"),
        Amount_MXN=("Amount MXN", "sum"),
        Assets=("ISIN", "count")
    )
    .sort_values("Weight", ascending=False)
)

pie_data["Weight %"] = pie_data["Weight"] * 100

pie_data["Hover_Text"] = (
    "<b>" + pie_data["Sleeve"].astype(str) + "</b><br>" +
    "Weight: " + pie_data["Weight %"].round(2).astype(str) + "%<br>" +
    "Amount MXN: $" + pie_data["Amount_MXN"].round(0).map("{:,.0f}".format) + "<br>" +
    "Assets: " + pie_data["Assets"].astype(int).astype(str)
)

fig = go.Figure(
    data=[
        go.Pie(
            labels=pie_data["Sleeve"],
            values=pie_data["Weight"],
            hole=0.35,
            textinfo="percent+label",
            textposition="inside",
            hoverinfo="text",
            hovertext=pie_data["Hover_Text"]
        )
    ]
)

fig.update_layout(
    title="Portfolio Allocation by Asset Type",
    width=900,
    height=650,
    legend_title_text="Asset Type"
)

fig.show()
display(pie_data)

,Sleeve,Weight,Amount_MXN,Assets,Weight %,Hover_Text
7,Mexican government fixed income,0.45,2.250000e+10,18,45.0,<b>Mexican government fixed income</b><br>Weig...
4,International equities,0.15,7.500000e+09,3,15.0,<b>International equities</b><br>Weight: 15.0%...
2,Domestic corporate and bank fixed income,0.12,6.000000e+09,30,12.0,<b>Domestic corporate and bank fixed income</b...
8,Structured instruments,0.10,5.000000e+09,2,10.0,<b>Structured instruments</b><br>Weight: 10.0%...
6,Mexican equities,0.08,4.000000e+09,2,8.0,<b>Mexican equities</b><br>Weight: 8.0%<br>Amo...
3,FIBRAs,0.03,1.500000e+09,2,3.0,<b>FIBRAs</b><br>Weight: 3.0%<br>Amount MXN: $...
5,International fixed income,0.03,1.500000e+09,2,3.0,<b>International fixed income</b><br>Weight: 3...
0,Cash and liquid assets,0.02,1.000000e+09,2,2.0,<b>Cash and liquid assets</b><br>Weight: 2.0%<...
1,Commodities,0.02,1.000000e+09,3,2.0,<b>Commodities</b><br>Weight: 2.0%<br>Amount M...


In [97]:
# ============================================================
# Barras interactivas: monto por sleeve
# Versión sin customdata para evitar errores de hovertemplate
# ============================================================

import plotly.graph_objects as go

bar_data = pie_data.copy()
bar_data["Amount MXN Billions"] = bar_data["Amount_MXN"] / 1e9

bar_data["Hover_Text"] = (
    "<b>" + bar_data["Sleeve"].astype(str) + "</b><br>" +
    "Amount: " + bar_data["Amount MXN Billions"].round(2).astype(str) + " billion MXN<br>" +
    "Weight: " + bar_data["Weight %"].round(2).astype(str) + "%<br>" +
    "Assets: " + bar_data["Assets"].astype(int).astype(str)
)

fig = go.Figure(
    data=[
        go.Bar(
            x=bar_data["Sleeve"],
            y=bar_data["Amount MXN Billions"],
            text=bar_data["Weight %"].round(1).astype(str) + "%",
            textposition="outside",
            hoverinfo="text",
            hovertext=bar_data["Hover_Text"]
        )
    ]
)

fig.update_layout(
    title="Portfolio Allocation Amount by Sleeve",
    xaxis_title="Asset Type",
    yaxis_title="Amount MXN Billions",
    width=1000,
    height=600
)

fig.show()

In [98]:
# ============================================================
# Backtesting: índice acumulado de cartera vs benchmark
# ============================================================

plot_bt = backtesting_table.reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_bt["Date"],
    y=plot_bt["Portfolio Cumulative Return Index"],
    mode="lines",
    name="Portfolio",
    hovertemplate="Date: %{x}<br>Portfolio Index: %{y:.3f}<extra></extra>"
))

if benchmark_ticker:
    fig.add_trace(go.Scatter(
        x=plot_bt["Date"],
        y=plot_bt["Benchmark Cumulative Return Index"],
        mode="lines",
        name=f"Benchmark ({benchmark_ticker})",
        hovertemplate="Date: %{x}<br>Benchmark Index: %{y:.3f}<extra></extra>"
    ))

fig.update_layout(
    title="Backtesting: Portfolio vs Benchmark",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    width=1000,
    height=600,
    hovermode="x unified"
)

fig.show()

In [99]:
# ============================================================
# Drawdown interactivo
# ============================================================

drawdown = portfolio_cum / portfolio_cum.cummax() - 1
drawdown_df = pd.DataFrame({"Date": drawdown.index, "Drawdown": drawdown.values})

fig = px.area(
    drawdown_df,
    x="Date",
    y="Drawdown",
    title="Portfolio Drawdown",
)

fig.update_traces(
    hovertemplate="Date: %{x}<br>Drawdown: %{y:.2%}<extra></extra>"
)

fig.update_layout(
    yaxis_tickformat=".0%",
    xaxis_title="Date",
    yaxis_title="Drawdown",
    width=1000,
    height=500
)

fig.show()

In [100]:
# ============================================================
# Distribución de retornos diarios con línea de VaR
# ============================================================

ret_df = pd.DataFrame({"Daily Return": portfolio_returns.dropna()})

fig = px.histogram(
    ret_df,
    x="Daily Return",
    nbins=60,
    title="Distribution of Portfolio Daily Returns"
)

fig.add_vline(
    x=-port_var_95_daily,
    line_dash="dash",
    annotation_text="VaR 95%",
    annotation_position="top left"
)

fig.update_traces(
    hovertemplate="Daily return bin: %{x:.2%}<br>Frequency: %{y}<extra></extra>"
)

fig.update_layout(
    xaxis_tickformat=".1%",
    xaxis_title="Daily Return",
    yaxis_title="Frequency",
    width=1000,
    height=550
)

fig.show()

In [101]:
# ============================================================
# Riesgo vs rendimiento por sleeve proxy
# ============================================================

sleeve_stats = []
for sleeve in sleeve_returns.columns:
    r = sleeve_returns[sleeve].dropna()
    sleeve_stats.append({
        "Sleeve": sleeve,
        "Annual Return": annualized_return(r),
        "Annual Volatility": annualized_volatility(r),
        "Weight": IPS_WEIGHTS.get(sleeve, np.nan),
        "Selected Proxy": selected_sleeve_proxy.get(sleeve, None)
    })

sleeve_stats = pd.DataFrame(sleeve_stats)
sleeve_stats["Weight %"] = sleeve_stats["Weight"] * 100

fig = px.scatter(
    sleeve_stats,
    x="Annual Volatility",
    y="Annual Return",
    size="Weight %",
    hover_name="Sleeve",
    hover_data={
        "Selected Proxy": True,
        "Weight %": ":.2f",
        "Annual Return": ":.2%",
        "Annual Volatility": ":.2%"
    },
    title="Risk / Return by Asset Sleeve Proxy"
)

fig.update_layout(
    xaxis_tickformat=".1%",
    yaxis_tickformat=".1%",
    xaxis_title="Annualized Volatility",
    yaxis_title="Annualized Return",
    width=950,
    height=600
)

fig.show()
display(sleeve_stats)

,Sleeve,Annual Return,Annual Volatility,Weight,Selected Proxy,Weight %
0,Mexican government fixed income,0.050992,0.024151,0.45,CETETRC.MX,45.0
1,Domestic corporate and bank fixed income,0.000000,0.000000,0.12,CORPTRCISHRS.MX,12.0
2,International fixed income,0.018606,0.059892,0.03,AGG,3.0
3,Mexican equities,0.113741,0.256632,0.08,EWW,8.0
4,International equities,0.138668,0.183101,0.15,ACWI,15.0
5,Structured instruments,0.091348,0.114490,0.10,AOR,10.0
6,FIBRAs,0.133602,0.314045,0.03,FUNO11.MX,3.0
7,Commodities,0.184362,0.170630,0.02,GLD,2.0
8,Cash and liquid assets,0.025730,0.002819,0.02,SHV,2.0


## 12. Tablas finales para entregar

In [102]:
# ============================================================
# Tablas finales
# ============================================================

final_assets_table = portfolio_assets[[
    "Sleeve", "Asset Name", "ISIN", "Tipo", "Emisora", "Serie",
    "Yield", "Yield Used", "Coupon", "Maturity", "Duration", "Price",
    "Rating", "Sector", "Weight %", "Amount MXN", "Yahoo Proxy", "Source"
]].copy()

final_proxy_table = selected_proxy_table.merge(proxy_table, on="Sleeve", how="left")

final_backtesting_table = backtesting_table.copy()

display(final_assets_table)
display(final_proxy_table)
display(final_backtesting_table.tail(20))
display(metrics)

# Exportar a Excel por si lo necesitas
OUTPUT_XLSX = "IPS_Portfolio_75_79_Output_Tables.xlsx"

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    final_assets_table.to_excel(writer, sheet_name="Selected Assets", index=False)
    final_proxy_table.to_excel(writer, sheet_name="Proxy Table", index=False)
    final_backtesting_table.to_excel(writer, sheet_name="Backtesting", index=True)
    metrics.to_excel(writer, sheet_name="Metrics", index=False)
    allocation_check.to_excel(writer, sheet_name="Allocation Check", index=False)

print(f"Archivo Excel exportado: {OUTPUT_XLSX}")

,Sleeve,Asset Name,ISIN,Tipo,Emisora,Serie,Yield,Yield Used,Coupon,Maturity,Duration,Price,Rating,Sector,Weight %,Amount MXN,Yahoo Proxy,Source
0,Cash and liquid assets,SPDR Bloomberg 1-3 Month T-Bill ETF,US78468R6633,ETF / Proxy,SPDR Bloomberg 1-3 Month T-Bill ETF,,NaN,0.050000,NaN,NaT,NaN,NaN,NaN,NaN,1.500000,7.500000e+08,BIL,Yahoo Finance proxy; optimized by historical r...
1,Cash and liquid assets,iShares Short Treasury Bond ETF,US4642886794,ETF / Proxy,iShares Short Treasury Bond ETF,,NaN,0.050000,NaN,NaT,NaN,NaN,NaN,NaN,0.500000,2.500000e+08,SHV,Yahoo Finance proxy; optimized by historical r...
2,Commodities,SPDR Gold Shares,US78463V1070,ETF / Proxy,SPDR Gold Shares,,NaN,0.030000,NaN,NaT,NaN,NaN,NaN,NaN,1.286290,6.431452e+08,GLD,Yahoo Finance proxy; optimized by historical r...
3,Commodities,iShares Silver Trust,US46428Q1094,ETF / Proxy,iShares Silver Trust,,NaN,0.030000,NaN,NaT,NaN,NaN,NaN,NaN,0.538690,2.693452e+08,SLV,Yahoo Finance proxy; optimized by historical r...
4,Commodities,United States Oil Fund,US91232N2071,ETF / Proxy,United States Oil Fund,,NaN,0.030000,NaN,NaT,NaN,NaN,NaN,NaN,0.175019,8.750960e+07,USO,Yahoo Finance proxy; optimized by historical r...
5,Domestic corporate and bank fixed income,BRHCCB 08-5U,MX97BR040092,97,BRHCCB,08-5U,5.985362,0.059854,6.55,NaT,NaN,35.979549,RETIRADA,SERVICIOS FINANCIEROS,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
6,Domestic corporate and bank fixed income,VWLEASE 24-2,MX91VW0100M4,91,VWLEASE,24-2,0.968545,0.968545,11.03,NaT,2.173560,105.319592,-,SERVICIOS FINANCIEROS,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
7,Domestic corporate and bank fixed income,GCDMXCB 18V,MX90GC040026,90,GCDMXCB,18V,0.995000,0.995000,9.93,NaT,2.182583,104.098188,-,SERVICIOS FINANCIEROS,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
8,Domestic corporate and bank fixed income,CFE 22-2S,MX95CF0500M1,95,CFE,22-2S,0.808868,0.808868,10.82,NaT,3.483811,106.777475,AAA(mex),SERVICIOS PÚBLICOS,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...
9,Domestic corporate and bank fixed income,ENGENCB 24-2,MX91EN090011,91,ENGENCB,24-2,0.020195,0.020195,12.98,NaT,2.982427,109.348493,AAA(mex),SERVICIOS FINANCIEROS,1.050584,5.252918e+08,NaN,Vector Analítico selection; optimized by yield...


,Sleeve,Selected Proxy,Status,Primary Proxy,Alternative Proxies,Explanation
0,Mexican government fixed income,CETETRC.MX,OK,CETETRC.MX,"M10TRACISHRS.MX, UDITRACISHRS.MX, SHV",Proxy for Mexican sovereign fixed income due t...
1,Domestic corporate and bank fixed income,CORPTRCISHRS.MX,OK,CORPTRCISHRS.MX,"LQD, AGG",Proxy for domestic corporate/bank debt; fallba...
2,International fixed income,AGG,OK,AGG,"BND, EMB",Proxy for global/U.S. aggregate and emerging-m...
3,Mexican equities,EWW,OK,EWW,"^MXX, MEXTRAC09.MX",Proxy for Mexican equity market exposure.
4,International equities,ACWI,OK,ACWI,"SPY, QQQ",Proxy for broad developed/global equity exposure.
5,Structured instruments,AOR,OK,AOR,"QQQ, SPY",Proxy for multi-asset/option-like structured p...
6,FIBRAs,FUNO11.MX,OK,FUNO11.MX,"FMTY14.MX, VNQ",Proxy for Mexican real estate trust exposure.
7,Commodities,GLD,OK,GLD,"SLV, USO",Proxy for liquid commodity exposures.
8,Cash and liquid assets,SHV,OK,SHV,"BIL, SGOV",Proxy for short-duration T-bills/cash equivale...


,Portfolio Daily Return,Portfolio Cumulative Return Index,Benchmark Ticker,Benchmark Daily Return,Benchmark Cumulative Return Index
Date,,,,,
2026-03-26,-0.007557,1.681474,ACWI,-0.021243,2.417505
2026-03-27,-0.003089,1.676280,ACWI,-0.013418,2.385066
2026-03-30,0.001556,1.678889,ACWI,-0.002676,2.378685
2026-03-31,0.010269,1.696130,ACWI,0.031150,2.452781
2026-04-01,0.003836,1.702637,ACWI,0.009395,2.475825
2026-04-02,-0.000890,1.701121,ACWI,-0.001647,2.471748
2026-04-06,0.001008,1.702835,ACWI,0.005092,2.484333
2026-04-07,0.000319,1.703379,ACWI,0.000357,2.485220
2026-04-08,0.010832,1.721830,ACWI,0.031526,2.563570


,Metric,Value
0,Expected / Backtested Annual Return,0.076078
1,Weighted Portfolio Yield,0.125424
2,Annualized Volatility,0.061559
3,Daily Historical VaR 95%,0.005395
4,Annualized Historical VaR 95%,0.085649
5,Sharpe Ratio,0.423619
6,Alpha vs Benchmark,0.029661
7,Beta vs Benchmark,0.310552
8,Maximum Drawdown,-0.138824
9,Benchmark,ACWI


Archivo Excel exportado: IPS_Portfolio_75_79_Output_Tables.xlsx


## 13. Interpretación corta

La cartera respeta el IPS de SIEFORE Básica Generacional 75–79 al mantener 60% en renta fija total, con 45% en instrumentos gubernamentales mexicanos obligatorios, 12% en deuda corporativa/bancaria local y 3% en renta fija internacional. El resto del portafolio se distribuye en renta variable mexicana e internacional, instrumentos estructurados, FIBRAs, commodities y liquidez. 

El backtesting usa proxies de Yahoo Finance porque muchos bonos individuales del Vector no tienen ticker directo disponible. Por eso, los proxies funcionan como representación del comportamiento histórico de cada sleeve, mientras que la selección específica de activos se conserva desde el Vector y desde la lista obligatoria de gobierno.